# 🐍 Day 11 · Python Basics — State Dictionaries & Graph Flow (LangGraph Basics)

> **Goal of this notebook:** LangGraph = a **state** (a dictionary) passed through **nodes** (functions) along **edges**. Build a mini graph engine in plain Python.

**How to use it**
1. Click on a grey code cell and press **Shift + Enter** to run it.
2. Read the comments (lines that start with `#`) — they explain every step.
3. Change the values and run the cell again. Breaking things is how you learn! 🙂

📦 **First** do *Step 1 - Install today's packages* in [`00_START_HERE.md`](00_START_HERE.md).

✅ No API key needed for this notebook.

### 📋 What you will learn
1. The state as a `TypedDict`
2. Nodes: functions that return updates
3. Merging dictionaries
4. Routing functions (conditional edges)
5. A tiny graph runner
6. Loops in a graph (generate → evaluate → improve)

## 1. The state
Every LangGraph workflow starts by describing its **state** — the information that travels through the graph.

In [ ]:
from typing import TypedDict, Literal

class ReviewState(TypedDict):
    review: str
    sentiment: str
    reply: str

state: ReviewState = {"review": "The battery is terrible", "sentiment": "", "reply": ""}
print(state)

## 2. Nodes — functions that return **only what changed**
A node receives the whole state and returns a small dictionary with the keys it updates.

In [ ]:
def find_sentiment(state):
    bad_words = ["terrible", "bad", "broken", "late"]
    is_bad = any(word in state["review"].lower() for word in bad_words)   # any() = True if at least one is True
    return {"sentiment": "negative" if is_bad else "positive"}

print(find_sentiment(state))

## 3. Merging dictionaries
LangGraph merges the node's update into the state. In Python: `{**old, **new}` or `old.update(new)`.

In [ ]:
old = {"review": "hi", "sentiment": "", "reply": ""}
update = {"sentiment": "positive"}

merged = {**old, **update}      # ** spreads the dictionaries; later keys win
print(merged)

## 4. Routing functions = conditional edges
A routing function looks at the state and returns the **name of the next node**.

In [ ]:
def positive_reply(state):
    return {"reply": "Thanks for the kind words! 🙏"}

def negative_reply(state):
    return {"reply": "We are sorry. A support agent will contact you."}

def route(state) -> Literal["positive_reply", "negative_reply"]:
    if state["sentiment"] == "positive":
        return "positive_reply"
    return "negative_reply"

print(route({"sentiment": "negative"}))

## 5. 🔗 A tiny graph runner
This is (a very small version of) what `graph.compile()` and `.invoke()` do for you.

In [ ]:
nodes = {
    "find_sentiment": find_sentiment,
    "positive_reply": positive_reply,
    "negative_reply": negative_reply,
}
edges = {
    "START": "find_sentiment",
    "find_sentiment": route,          # a function = conditional edge
    "positive_reply": "END",
    "negative_reply": "END",
}

def invoke(state):
    current = edges["START"]
    while current != "END":
        print("➡️  running node:", current)
        state = {**state, **nodes[current](state)}   # run node + merge update
        nxt = edges[current]
        current = nxt(state) if callable(nxt) else nxt
    return state

print(invoke({"review": "The battery is terrible", "sentiment": "", "reply": ""}))
print(invoke({"review": "I love the camera", "sentiment": "", "reply": ""}))

## 6. Loops in a graph
`04_X_post_generator_workflow.ipynb` writes a post, **evaluates** it, and improves it until it is good — but at most N times.

In [ ]:
def generate(state):
    attempt = state["attempt"] + 1
    post = "AI is cool" + "!" * attempt        # pretend each try gets better
    return {"post": post, "attempt": attempt}

def evaluate(state):
    ok = state["post"].count("!") >= 3          # our "quality" rule
    return {"approved": ok}

state = {"post": "", "attempt": 0, "approved": False}
max_attempts = 5
while not state["approved"] and state["attempt"] < max_attempts:
    state.update(generate(state))
    state.update(evaluate(state))
    print(state)

---
## 🏋️ Practice time
Try these on your own in the empty cell below:

1. Add a word to `bad_words` (for example `"slow"`) and test `invoke` with `"The app is slow"`.
2. Write a node `count_words(state)` that returns `{"words": len(state["review"].split())}`.
3. Merge `{"a": 1, "b": 2}` and `{"b": 99}`. What is the value of `b`?

In [ ]:
# ✍️ Write your answers here

<details>
<summary>👀 Click here to see the solutions (try first!)</summary>

```python
def count_words(state):
    return {"words": len(state["review"].split())}
print(count_words({"review": "nice phone overall"}))

print({**{"a": 1, "b": 2}, **{"b": 99}})   # b is 99 - the later dict wins
```
</details>

## ✅ Summary

| Concept | Example |
|---|---|
| State | `class State(TypedDict): review: str` |
| Node | `def node(state): return {'key': value}` |
| Merge dicts | `{**old, **new}` |
| Any is true | `any(w in text for w in words)` |
| Router | `def route(state): return 'next_node'` |
| Is it a function? | `callable(x)` |

## ➡️ What's next?
You now know the Python used in **Day_11_LangGraph_Basics_Workflows**.

Open [`00_START_HERE.md`](00_START_HERE.md) and continue with **lesson 1**. 🚀